# [Chapter 6 — SIR_I Analysis, §6.2] The Basic Reproductive Number — Heuristic Derivation

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 6 — SIR_I Analysis
**Considerations developed:** 6 (R_0)
**Estimated runtime:** ≤ 15 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Derives $R_0 = c_I \beta \tau_R$ heuristically: how many secondary infections does a single infectious individual produce in a fully susceptible population? Verifies the heuristic against the early-outbreak growth rate from a numerical simulation.

## What you should already know
Chapter 5 notebooks.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_07
from shared.seeds import set_seed_chapter_06
from shared.verification import assert_within_tolerance

set_seed_chapter_06()
book_style()


## The heuristic

In a fully susceptible population ($S/N \approx 1$), one infectious individual:
1. Makes $c_I$ contacts per day
2. Each contact has probability $\beta$ of transmitting (since $S/N = 1$, every contact is with a susceptible)
3. Remains infectious for $\tau_R$ days on average

Therefore the expected number of secondary infections is:

$$R_0 = c_I \cdot \beta \cdot \tau_R$$

This is the heuristic derivation. Chapter 6's NGM notebook gives the rigorous derivation.

## Verification via early-outbreak growth rate

In the linearized regime ($S \approx N$), the SIR_I system reduces to:
$$\frac{dI}{dt} = (\alpha_0 - 1/\tau_R) I = (R_0 - 1) \frac{I}{\tau_R}$$

So $I(t)$ grows exponentially with rate $r = (R_0 - 1)/\tau_R$. Measuring $r$ and inverting gives $R_0$.

In [ ]:
params = baseline_chapter_07()
result = integrate_sir_i(params)
t = result['t']
I = result['I']

R0_predicted = params['c_I'] * params['beta'] * params['tau_R']
r_predicted = (R0_predicted - 1) / params['tau_R']

# Fit log(I) vs t over early outbreak (first 20 days, before substantial S depletion)
mask = (t > 5) & (t < 20)
log_I_early = np.log(I[mask])
slope, intercept = np.polyfit(t[mask], log_I_early, 1)

R0_measured = 1 + slope * params['tau_R']

print(f"Heuristic prediction: R_0 = c_I * beta * tau_R = {R0_predicted:.4f}")
print(f"Predicted growth rate: r = (R_0 - 1)/tau_R = {r_predicted:.4f} per day")
print()
print(f"Measured growth rate (linear fit to log I): r = {slope:.4f} per day")
print(f"Inferred R_0 from measured growth: {R0_measured:.4f}")
print()

# Verify within tolerance
assert_within_tolerance(R0_measured, R0_predicted, rel_tol=0.05)
print("✅ Verified: heuristic R_0 agrees with simulated growth rate within 5%.")


In [ ]:
# Visualize the linearized growth phase
fig, ax = plt.subplots(figsize=(7, 4.2))
mask_plot = t < 30
ax.semilogy(t[mask_plot], I[mask_plot], color=BOOK_COLORS['infectious'], lw=1.5, label='I(t) (simulated)')

t_fit = t[mask]
I_fit_predicted = np.exp(intercept + slope * t_fit)
ax.semilogy(t_fit, I_fit_predicted, color=BOOK_COLORS['neutral'], ls='--', lw=1.2, label=f'Linear fit: $r = {slope:.3f}$/day')

ax.set_xlabel('Time (days)')
ax.set_ylabel('I(t) (log scale)')
ax.set_title(f'Early-outbreak exponential growth $\\Rightarrow$ $R_0 = {R0_measured:.2f}$')
ax.legend()
plt.tight_layout()
plt.show()
